# 01 — BePLi v2: download, inspect, leakage audit, splits (Colab)
Milestone 3. **Run on Google Colab (free). Do not run this on the Mac: the archive is ~8 GB.**
Nothing in this notebook has been executed yet on real data. Record real outputs in `reports/` and `HANDOFF.md`.

Before running: copy the project folder to Google Drive (e.g. `MyDrive/marine-plastic/`). Pick a CPU runtime (no GPU needed).

## 1. Mount Drive, go to the project, install the few extra packages

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/marine-plastic'   # <- adjust if you put it elsewhere
%cd $PROJECT
!pip -q install pycocotools==2.0.8 PyYAML==6.0.2
import sys, os; sys.path.insert(0, PROJECT)
import yaml; cfg = yaml.safe_load(open('configs/data.yaml')); cfg

## 2. Download + extract on Colab's local disk (NOT Drive)
Open https://www.seanoe.org/data/00858/96963/ , right-click the Download button, **Copy link address**, and paste it below.
The script refuses to run if less than 25 GB is free, records size and SHA-256, and extracts safely.

In [ ]:
URL = ''   # <- paste the copied direct link here
assert URL, 'paste the download link first'
!python -m ml.data.fetch_bepli --url "$URL" --dest /content/bepli_v2 --min-free-gb 25

## 3. Find the files and inspect them (real counts, classes, problems, claimed-vs-measured)

In [ ]:
!find /content/bepli_v2/raw -maxdepth 3 | head -40
c = cfg['dataset']['claimed']
!python -m ml.data.inspect_coco --root /content/bepli_v2/raw --out reports/dataset_inspection \
    --sample-decode {cfg['inspection']['sample_decode']} --expect-images {c['images']} --expect-classes {c['classes']} --expect-annotations {c['annotations']}

In [ ]:
print(open('reports/dataset_inspection.md').read()[:6000])

## 4. LOOK at the data (required manual step)
View ~30 random images with their masks. Answer in `docs/DATASET_CARD.md`: Are all plastic items annotated? Are tiny fragments missed? Are the views from different directions of the same place?

In [ ]:
import json, random, glob
from PIL import Image, ImageDraw
from ml.data.coco_utils import find_coco_files, load_json, build_image_index, resolve_image
root = '/content/bepli_v2/raw'
ann_path = find_coco_files(root)[0]; data = load_json(ann_path); idx = build_image_index(root)
by_img = {}
for a in data['annotations']: by_img.setdefault(a['image_id'], []).append(a)
sample = random.Random(0).sample(data['images'], 6)
import matplotlib.pyplot as plt
fig, axs = plt.subplots(2, 3, figsize=(15, 8))
for ax, im in zip(axs.ravel(), sample):
    p = resolve_image(root, ann_path, im['file_name'], idx)
    img = Image.open(p).convert('RGB'); d = ImageDraw.Draw(img)
    for a in by_img.get(im['id'], []):
        x, y, w, h = a['bbox']; d.rectangle([x, y, x + w, y + h], outline=(255, 0, 0), width=2)
    ax.imshow(img); ax.set_title(f"{im['file_name']}  ({len(by_img.get(im['id'], []))} objects)"); ax.axis('off')
plt.show()

## 5. Leakage audit of the dataset's OWN split files (if it ships train/val/test json files)
Pass one `--ann` per split file found in step 3. If there is only one json file, skip to step 6.

In [ ]:
# EDIT the list to the split files found in step 3, e.g. ['/content/bepli_v2/raw/.../train.json', ...]
ANN = []
if ANN:
    args = ' '.join(f'--ann {a}' for a in ANN)
    !python -m ml.data.near_duplicates --root /content/bepli_v2/raw {args} --threshold {cfg['leakage']['dup_threshold']} --out reports/leakage_audit.json

## 6. Build OUR leakage-aware split, then verify it
`block_size` stays 0 until you have checked whether consecutive file numbers belong to the same site visit (step 4).

In [ ]:
ALL = ['/content/bepli_v2/raw/PATH/TO/annotations.json']   # <- EDIT: all annotation files to pool
args = ' '.join(f'--ann {a}' for a in ALL)
!python -m ml.data.make_splits --root /content/bepli_v2/raw {args} --out data/splits/bepli_v2_grouped.json \
    --ratios 0.7,0.15,0.15 --seed {cfg['split']['seed']} --dup-threshold {cfg['leakage']['dup_threshold']} --block-size {cfg['leakage']['block_size']}
!python -m ml.data.check_leakage --root /content/bepli_v2/raw {args} --splits data/splits/bepli_v2_grouped.json --threshold {cfg['leakage']['dup_threshold']}

## 7. Tile-label statistics (positive / negative / uncertain) on a sample
Check the three counts look sensible and view a few tiles of each kind before freezing the rules.

In [ ]:
import numpy as np
from collections import Counter
from ml.data import tile_labels as tl
T = cfg['tiles']; counts = Counter()
for im in random.Random(1).sample(data['images'], min(300, len(data['images']))):
    m = tl.union_mask(by_img.get(im['id'], []), im['height'], im['width'])
    boxes = tl.tile_boxes(im['height'], im['width'], T['tile'], T['stride'])
    labels, _ = tl.label_tiles(m, boxes, T['pos_frac'], T['margin'])
    counts.update(labels.tolist())
print({'positive': counts[1], 'negative': counts[0], 'uncertain': counts[-1]})

## 8. Compact copy for training (JPEG, ~1-2 GB expected; UNVERIFIED size) — saved to Drive
Only run after the splits are frozen. Re-encoding is lossy: state it in the dataset card.

In [ ]:
OUT = '/content/drive/MyDrive/marine-plastic-data/bepli_compact'
for a in ALL:
    !python -m ml.data.prepare_subset --root /content/bepli_v2/raw --ann {a} --out {OUT} --quality 95
!du -sh {OUT}

## 9. Finish
1. Copy `reports/*.json|md`, `data/splits/*.json` back to the project on your Mac (Drive sync or download).
2. Paste the real numbers into `docs/DATASET_CARD.md` and `HANDOFF.md` (replace every UNVERIFIED mark you have now verified).
3. Tell Claude/ChatGPT the results of steps 3, 4, 5, 6.